# Word2Vec

## Skip-gram とネガティブ サンプリング

Bag-of-words モデルが周囲のコンテキストが与えられたときに単語を予測するのに対し、Skip-gram モデルは単語そのものが与えられたときにそのコンテキスト（周囲の単語）を予測します。

次の 8 単語からなる文を考えてみましょう。
> The wide road shimmered in the hot sun.

ウィンドウ サイズを 2 とした場合、8 単語それぞれのコンテキスト ワードは次のようになります。

- (wide, road) = context(The)
- (The, road, shimmered) = context(wide)
- (The, wide, shimmered, in) = context(road)
- (wide, road, in, the) = context(shimmered)
- (road, shimmered, the, hot) = context(in)
- (shimmered, in, hot, sun) = context(the)
- (in, the, sun) = context(hot)
- (the, hot) = context(sun)

Skip-gram は、ターゲット ワードとコンテキスト ワードのペア（例: (road, in)、(road, the)）を考慮します。このとき、`(road, in)` は正例（positive）のペアとなります。

注: このラボでは、ウィンドウ サイズ *n* は各側に n 個の単語があることを意味し、合計のウィンドウ サイズは 2*n+1 個の単語になります。

![word2vec_skipgrams](assets/word2vec_skipgram.png)

Skip-gram モデルのトレーニング目標は、ターゲット ワードが与えられたときにコンテキスト ワードを予測する確率を最大化することです。単語のシーケンス *w<sub>1</sub>, w<sub>2</sub>, ... w<sub>T</sub>* に対して、目標は平均対数確率を最大化することです。

![word2vec_skipgram_objective](assets/word2vec_skipgram_objective.png)

ここで `c` はトレーニング コンテキストのサイズです。基本的な Skip-gram の定式化では、ソフトマックス関数を使用してこの確率を定義します。

![word2vec_full_softmax](assets/word2vec_full_softmax.png)

ここで *v* と *v<sup>'</sup>* は単語のターゲットおよびコンテキストのベクトル表現であり、*W* は語彙サイズです。

この定式化の分母を計算するには、語彙全体の単語に対して完全なソフトマックスを実行する必要があり、語彙サイズは多くの場合非常に大きくなります（10<sup>5</sup>〜10<sup>7</sup> 語）。

[雑音対照推定（Noise Contrastive Estimation: NCE）](https://www.tensorflow.org/api_docs/python/tf/nn/nce_loss) 損失関数は、完全なソフトマックスの効率的な近似手法です。単語の分布をモデル化するのではなく単語の埋め込みを学習することを目標とする場合、NCE 損失はネガティブ サンプリング（Negative Sampling）に [単純化](https://papers.nips.cc/paper/5021-distributed-representations-of-words-and-phrases-and-their-compositionality.pdf) できます。

ターゲット ワードに対する単純化されたネガティブ サンプリングの目標は、ノイズ分布から抽出された *num_ns* 個のネガティブ サンプル（負例）からコンテキスト ワード（正例）を識別することです。より数学的には、目標は以下の条件付き確率の対数尤度を最大化することです。

$$Q_\theta(D=1 | \text{word}, \text{context}) = \sigma(\mathbf{v}^\prime_\text{context} \mathbf{v}_\text{word})$$

ここで、$\sigma$ はシグモイド関数です。

ネガティブ サンプルは、`context_word` が `target_word` の `window_size` 近傍に現れないような `(target_word, context_word)` ペアとして定義されます。

次のセクションでは、メカニズムを理解するために、単一の文に対して Skip-gram とネガティブ サンプルを生成します。

## セットアップ

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# バックエンドを選択: "tensorflow", "jax", または "torch"
os.environ["KERAS_BACKEND"] = "tensorflow"
import io
import re
import string

import keras
import numpy as np
import tensorflow as tf  # tf.data パイプライン用に保持
import tqdm
from keras import layers, ops  # ops にはバックエンド非依存の数学関数が含まれます

下のセルを使用して、TensorFlow と Keras のバージョンを確認してください。

In [ ]:
print(f"TensorFlow version: {tf.version.VERSION}")
print(f"Keras version: {keras.__version__}")

In [ ]:
# 必要に応じて以下を変更してください
PROJECT = !gcloud config get-value project  # noqa: E999
PROJECT = PROJECT[0]
BUCKET = PROJECT
REGION = "us-central1"

OUTDIR = f"gs://{BUCKET}/text_models"

%env PROJECT=$PROJECT
%env BUCKET=$BUCKET
%env REGION=$REGION
%env OUTDIR=$OUTDIR

In [ ]:
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

### サンプル文のベクトル化

次の文を考えてみます。
    `The wide road shimmered in the hot sun`
文内の単語（トークン）:

In [ ]:
sentence = "The wide road shimmered in the hot sun"
tokens = list(sentence.lower().split())
print(len(tokens))

トークンから整数インデックスへのマッピングを保存するための語彙（vocabulary）を作成します。

In [ ]:
vocab, index = {}, 1  # インデックスは 1 から開始
vocab["<pad>"] = 0  # パディング トークンを追加
for token in tokens:
    if token not in vocab:
        vocab[token] = index
        index += 1
vocab_size = len(vocab)
print(vocab)

整数インデックスからトークンへの逆マッピングを保存するための逆語彙を作成します。

In [ ]:
inverse_vocab = {index: token for token, index in vocab.items()}
print(inverse_vocab)

文をベクトル化します。

In [ ]:
example_sequence = [vocab[word] for word in tokens]
print(example_sequence)

### 1 つの文から Skip-gram を生成する

`tf.keras.preprocessing.sequence` モジュールは、Word2Vec のデータ準備を簡素化する便利な関数を提供しています。`tf.keras.preprocessing.sequence.skipgrams` を使用して、指定したシーケンスから、`[0, vocab_size)` の範囲のトークンと指定した `window_size` を用いて Skip-gram ペアを生成できます。

In [ ]:
window_size = 2
positive_skip_grams, _ = tf.keras.preprocessing.sequence.skipgrams(
    example_sequence,
    vocabulary_size=vocab_size,
    window_size=window_size,
    negative_samples=0,
    seed=SEED
)
print(len(positive_skip_grams))

生成された正例の Skip-gram をいくつか確認してみましょう。

In [ ]:
for target, context in positive_skip_grams[:5]:
    print(
        f"({target}, {context}): ({inverse_vocab[target]}, {inverse_vocab[context]})"
    )

### 1 つの Skip-gram に対するネガティブ サンプリング

`skipgrams` 関数は、指定されたウィンドウ境界をスライドすることで、すべての正例の Skip-gram ペアを返します。トレーニング用の追加のネガティブ サンプルを生成するには、語彙からランダムな単語をサンプリングする必要があります。`tf.random.log_uniform_candidate_sampler` 関数を使用すると、ウィンドウ内の特定のターゲット ワードに対して `num_ns` 個のネガティブ アイテムをサンプリングできます。

重要なポイント: 正例のコンテキスト ワードあたりのネガティブ サンプル数（*num_ns*）は、小規模なデータセットでは [5, 20] の間が最も効果的であり、大規模なデータセットでは [2, 5] で十分であることが [示されています](https://papers.nips.cc/paper/5021-distributed-representations-of-words-and-phrases-and-their-compositionality.pdf)。

In [ ]:
# 1 つの正例 Skip-gram からターゲットとコンテキストの単語を取得
target_word, context_word = positive_skip_grams[0]

# 正例コンテキストあたりのネガティブ サンプル数を設定
num_ns = 4

context_class = ops.reshape(
    ops.convert_to_tensor(context_word, dtype="int64"), (1, 1)
)
negative_sampling_candidates, _, _ = tf.random.log_uniform_candidate_sampler(
    true_classes=context_class,  # '正例' としてサンプリングされるべきクラス
    num_true=1,  # 各正例 Skip-gram は 1 つの正例コンテキスト クラスを持つ
    num_sampled=num_ns,  # サンプリングするネガティブ コンテキスト ワードの数
    unique=True,  # すべてのネガティブ サンプルは一意である必要がある
    range_max=vocab_size,  # サンプルのインデックスを [0, vocab_size] から選択
    seed=SEED,  # 再現性のためのシード
    name="negative_sampling",  # この操作の名前
)
print(negative_sampling_candidates)
print([inverse_vocab[index.numpy()] for index in negative_sampling_candidates])

### 1 つのトレーニング サンプルの構築

指定された正例の `(target_word, context_word)` Skip-gram に対して、`num_ns` 個のネガティブ サンプリングされたコンテキスト ワードも得られました。正例のコンテキスト ワードとネガティブ サンプリングされたコンテキスト ワードを連結してラベルを生成できます。最初のコンテキスト ワードは 1（正例）とラベル付けされ、それに続いて `num_ns` 個の 0（負例）が並びます。

In [ ]:
# 次のステップで連結を使用できるように次元を追加
negative_sampling_candidates = tf.expand_dims(negative_sampling_candidates, 1)

# 正例のコンテキスト ワードとネガティブ サンプリングされた単語を連結
context = tf.concat([context_class, negative_sampling_candidates], 0)

# 最初のコンテキスト ワードを 1（正例）とラベル付けし、続いて num_ns 個の 0（負例）を付与
label = tf.constant([1] + [0] * num_ns, dtype="int64")

# target を shape (1,) に、context と label を (num_ns+1,) に整形
target = tf.squeeze(target_word)
context = tf.squeeze(context)
label = tf.squeeze(label)


# 次のステップで連結を使用できるように次元を削減
squeezed_context_class = ops.squeeze(context_class, 1)

# 正例のコンテキスト ワードとネガティブ サンプリングされた単語を連結
context = ops.concatenate(
    [squeezed_context_class, negative_sampling_candidates], 0
)

# 最初のコンテキスト ワードを `1`（正例）とラベル付けし、続いて `num_ns` 個の `0`（負例）を付与
label = ops.convert_to_tensor([1] + [0] * num_ns, dtype="int64")
target = target_word

上記の Skip-gram から、ターゲット ワードのコンテキストと対応するラベルを確認してみましょう。

In [ ]:
print(f"target_index    : {target}")
print(f"target_word     : {inverse_vocab[target_word]}")
print(f"context_indices : {context}")
print(f"context_words   : {[inverse_vocab[c.numpy()] for c in context]}")
print(f"label           : {label}")

`(target, context, label)` の Tensor のタプルが、Skip-gram ネガティブ サンプリング Word2Vec モデルをトレーニングするための 1 つのトレーニング サンプルを構成します。ターゲットの shape は `(1,)` であり、コンテキストとラベルの shape は `(1+num_ns,)` であることに注目してください。

In [ ]:
print(f"target  :", target)
print(f"context :", context)
print(f"label   :", label)

### まとめ

この図は、文からトレーニング サンプルを生成する手順をまとめたものです。

![word2vec_negative_sampling](assets/word2vec_negative_sampling.png)

## ラボ課題 1

### Skip-gram サンプリング テーブル

データセットが大きいということは、"the"、"is"、"of" などのストップワードのような高頻度語を多く含む、より大きな語彙を意味します。このような単語はモデルにほとんどコンテキスト情報を提供しません。Word2Vec はトレーニングのバランスを取るために高頻度語をサブサンプリングします。

`tf.keras.preprocessing.sequence.skipgrams` 関数は、任意のトークンをサンプリングする確率をエンコードするサンプリング テーブル引数を受け入れます。`tf.keras.preprocessing.sequence.make_sampling_table` を使用して、単語頻度の順位に基づいた確率的サンプリング テーブルを生成し、それを `skipgrams` 関数に渡すことができます。

In [ ]:
sampling_table = tf.keras.preprocessing.sequence.make_sampling_table(size=10)
print(sampling_table)

`sampling_table[i]` は、データセット内で i 番目に一般的な単語をサンプリングする確率を示します。この関数は、サンプリングのための単語頻度が [ジップの法則（Zipf's law）](https://en.wikipedia.org/wiki/Zipf%27s_law) に従うことを前提としています。

重要なポイント: `tf.random.log_uniform_candidate_sampler` は、語彙の頻度が対数一様（Zipf）分布に従うことをすでに前提としています。これらの分布ベースのサンプリング テーブルを使用することは、モデルのネガティブ サンプリング目標を簡素化するのにも役立ちます。

### トレーニング データの生成

上記で説明したすべての手順を、任意のテキスト データセットのベクトル化された文のリストに対して呼び出せる関数にまとめます。Skip-gram ペアを生成する際にサンプリング テーブルが使用されていることに注目してください。

**ラボ課題 1**

1a - 以下のコードで、次を完成させてください。
1. `vocab_size` 個のトークンに対するサンプリング テーブルを構築します。
2. データセット内の各シーケンスを反復処理します。
3. `skipgrams` を呼び出して正例の Skip-gram ペアを生成します。

1b - 以下のコードで、次を完成させてください。
1. 各正例 Skip-gram ペアからコンテキスト ワードを抽出します。
2. `tf.random.log_uniform_candidate_sampler` を使用して `num_ns` 個のネガティブ ワードをサンプリングします。
3. 正例と負例のコンテキスト ワードを連結します。
4. ラベル ベクトルを連結します。

In [ ]:
"""
Generates skip-gram pairs with negative sampling for a list of sequences
(int-encoded sentences) based on window size, number of negative samples
and vocabulary size.
"""


def generate_training_data(sequences, window_size, num_ns, vocab_size, seed):
    # 各トレーニング サンプルの要素がこれらのリストに追加されます
    targets, contexts, labels = [], [], []

    # vocab_size 個のトークンに対するサンプリング テーブルを構築
    # TODO 1a
    sampling_table = tf.keras.preprocessing.sequence.make_sampling_table(
        vocab_size
    )

    # データセット内のすべてのシーケンス（文）を反復処理
    # TODO 1a: ここにコードを記述してください

        # 各正例 Skip-gram ペアを反復処理して、正例コンテキスト ワードと
        # ネガティブ サンプルを含むトレーニング サンプルを生成
        # TODO 1b
        for target_word, context_word in positive_skip_grams:
            context_class = # TODO 1b: ここにコードを記述してください

            # コンテキスト ベクトルとラベル ベクトルを構築（1 つのターゲット ワード用）
            context = ops.concatenate(
                [
                    ops.squeeze(context_class, axis=1),
                    negative_sampling_candidates,
                ],
                0,
            )
            label = ops.convert_to_tensor([1] + [0] * num_ns, dtype="int64")

            # トレーニング サンプルの各要素をグローバル リストに追加
            targets.append(target_word)
            contexts.append(context)
            labels.append(label)

    return targets, contexts, labels

## ラボ課題 2: Word2Vec 用のトレーニング データの準備

Skip-gram ネガティブ サンプリング Word2Vec モデル用に 1 つの文を処理する方法を理解したところで、より大きな文のリストからトレーニング サンプルを生成する手順に進みましょう。

### テキスト コーパスのダウンロード

このチュートリアルでは、シェイクスピアの著作のテキスト ファイルを使用します。他のテキスト データセットでこのコードを実行するには、次の行を変更してください。

In [ ]:
path_to_file = keras.utils.get_file(
    "shakespeare.txt",
    "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt",
)

ファイルからテキストを読み込み、最初の数行を確認します。

In [ ]:
with open(path_to_file) as f:
    lines = f.read().splitlines()
for line in lines[:20]:
    print(line)

**ラボ課題 2**
空でない行を使用して、後続のステップで使用する `tf.data.TextLineDataset` オブジェクトを構築してください。

In [ ]:
# TODO 2a
text_ds = # ここにコードを記述してください

### コーパスからの文のベクトル化

`TextVectorization` レイヤーを使用して、コーパスの文をベクトル化できます。このレイヤーの使用方法の詳細については、この [テキストの前処理（Text Preprocessing）](https://www.tensorflow.org/tutorials/keras/text_classification) チュートリアルを参照してください。

In [ ]:
"""
We create a custom standardization function to lowercase the text and
remove punctuation.
"""


def custom_standardization(input_data):
    lowercase = tf.strings.lower(input_data)
    return tf.strings.regex_replace(
        lowercase, "[%s]" % re.escape(string.punctuation), ""
    )


"""
Define the vocabulary size and number of words in a sequence.
"""
vocab_size = 4096
sequence_length = 10

"""
Use the text vectorization layer to normalize, split, and map strings to
integers. Set output_sequence_length length to pad all samples to same length.
"""
vectorize_layer = layers.TextVectorization(
    standardize=custom_standardization,
    max_tokens=vocab_size,
    output_mode="int",
    output_sequence_length=sequence_length,
)

テキスト データセットに対して `adapt` を呼び出して語彙を作成します。

In [ ]:
vectorize_layer.adapt(text_ds.batch(1024))

レイヤーの状態がテキスト コーパスを表すように適応されたら、`get_vocabulary()` で語彙にアクセスできます。この関数は、頻度順にソートされたすべての語彙トークンのリストを返します（最も頻度の高いトークンが最初）。

In [ ]:
# 参照用に作成した語彙を保存
inverse_vocab = vectorize_layer.get_vocabulary()
print(inverse_vocab[:20])

これで、`vectorize_layer` を使用して `text_ds`（`tf.data.Dataset`）内の各要素のベクトルを生成できるようになりました。

In [ ]:
# text_ds 内のデータをベクトル化
text_vector_ds = (
    text_ds.batch(1024).prefetch(AUTOTUNE).map(vectorize_layer).unbatch()
)

### データセットからのシーケンスの取得

これで、整数エンコードされた文の `tf.data.Dataset` が得られました。`generate_training_data()` 関数で処理できるようにデータセットを準備するには、データセットを文ベクトルのシーケンスのリストに変換します。各要素を反復処理してデータセットを走査し、文ベクトルの各 `tf.Tensor` を整数の Python リストに変換します。

In [ ]:
sequences = list(text_vector_ds.as_numpy_iterator())
print(len(sequences))

`sequences` からいくつかのサンプルを確認してみましょう。

In [ ]:
for seq in sequences[:5]:
    print(f"{seq} => {[inverse_vocab[i] for i in seq]}")

### シーケンスからのトレーニング サンプルの生成

`sequences` は整数エンコードされた文のリストになりました。先ほど定義した `generate_training_data` 関数を呼び出して、Word2Vec モデル用のトレーニング サンプルを生成します。復習すると、この関数は各シーケンスの各単語を反復処理して、正例と負例のサンプルを生成します。この実行には少し時間がかかります。

In [ ]:
targets, contexts, labels = generate_training_data(
    sequences=sequences,
    window_size=2,
    num_ns=4,
    vocab_size=vocab_size,
    seed=SEED,
)

targets = np.array(targets)
contexts = np.array(contexts)
labels = np.array(labels)

print("\n")
print(f"targets.shape: {targets.shape}")
print(f"contexts.shape: {contexts.shape}")
print(f"labels.shape: {labels.shape}")

### パフォーマンスのためのデータセットの設定

潜在的に膨大な数のトレーニング サンプルに対して効率的なバッチ処理を実行するために、`tf.data.Dataset` API を使用します。このステップの後、Word2Vec モデルをトレーニングするための `(target_word, context_word), (label)` のバッチ処理 `tf.data.Dataset` オブジェクトが得られます。

In [ ]:
BATCH_SIZE = 1024
BUFFER_SIZE = 10000
dataset = tf.data.Dataset.from_tensor_slices(((targets, contexts), labels))
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)
print(dataset)

パフォーマンスを向上させるために `cache()` と `prefetch()` を追加します。

In [ ]:
dataset = dataset.cache().prefetch(buffer_size=AUTOTUNE)
print(dataset)

## ラボ課題 3: モデルとトレーニング

Word2Vec モデルは、Skip-gram からの真のコンテキスト ワードとネガティブ サンプリングによって得られた偽のコンテキスト ワードを区別する分類器として実装できます。ターゲット ワードとコンテキスト ワードの埋め込みの間でドット積（内積）を実行してラベルの予測を取得し、データセット内の真のラベルに対して損失関数を計算できます。

### サブクラス化された Word2Vec モデル

[Keras Subclassing API](https://keras.io/guides/making_new_layers_and_models_via_subclassing/) を使用して、次のレイヤーを持つ Word2Vec モデルを定義します。

* `target_embedding`: ターゲット ワードがモデルに入力されたときにその埋め込みを検索する `keras.layers.Embedding` レイヤー。このレイヤーのパラメータ数は `(vocab_size * embedding_dim)` です。
* `context_embedding`: コンテキスト ワードの埋め込みを検索するもう 1 つの `keras.layers.Embedding` レイヤー。このレイヤーのパラメータ数も `(vocab_size * embedding_dim)` です。
* `dots`: ターゲットとコンテキストの埋め込みのドット積を計算する `keras.ops.einsum`。
* `flatten`: ドット積の結果をロジットに平坦化する `keras.layers.Flatten` レイヤー。

重要なポイント: `target_embedding` と `context_embedding` のレイヤーは共有することもできます。また、両方の埋め込みの連結を最終的な Word2Vec 埋め込みとして使用することもできます。

In [ ]:
class Word2Vec(keras.Model):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.target_embedding = layers.Embedding(
            vocab_size, embedding_dim, name="w2v_embedding"
        )
        self.context_embedding = layers.Embedding(vocab_size, embedding_dim)

    def call(self, pair):
        target, context = pair
        if len(target.shape) == 2:
            target = ops.squeeze(target, axis=1)

        word_emb = self.target_embedding(target)
        context_emb = self.context_embedding(context)

        # バックエンド非依存のドット積に ops.einsum を使用
        # (batch, embed), (batch, context, embed) -> (batch, context)
        dots = ops.einsum("be,bce->bc", word_emb, context_emb)
        return dots

### 損失関数の定義とモデルのコンパイル

簡単にするため、ネガティブ サンプリング損失の代替として `keras.losses.CategoricalCrossentropy` を使用できます。独自のカスタム損失関数を作成したい場合は、次のように記述することもできます。

```python
def custom_loss(x_logit, y_true):
      return tf.nn.sigmoid_cross_entropy_with_logits(logits=x_logit, labels=y_true)
```

In [ ]:
# TODO 3a
embedding_dim = 128
word2vec = Word2Vec(vocab_size, embedding_dim)
# ここにコードを記述してください

TensorBoard 用のトレーニング統計をログに記録するためのコールバックも定義します。

In [ ]:
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir="logs")

上記で準備した `dataset` を使用して、指定したエポック数でモデルをトレーニングします。

In [ ]:
dataset

In [ ]:
word2vec.fit(dataset, epochs=50, callbacks=[tensorboard_callback])

### TensorBoard によるトレーニングの可視化

トレーニングの進行状況を監視するために、TensorBoard を使用して損失メトリクスを可視化できます。

In [ ]:
def copy_tensorboard_logs(local_path: str, gcs_path: str):
    """TensorBoard のログをローカル ディレクトリから GCS の場所にコピーします。
    トレーニング後、ローカルの TensorBoard ログを GCS の場所に一括コピーします。
    引数:
      local_path: ローカル ファイルシステムのディレクトリ URI。
      gcs_path: クラウド ファイルシステムのディレクトリ URI。
    戻り値:
      None。
    """
    pattern = f"{local_path}/*/events.out.tfevents.*"
    local_files = tf.io.gfile.glob(pattern)
    gcs_log_files = [
        local_file.replace(local_path, gcs_path) for local_file in local_files
    ]
    for local_file, gcs_file in zip(local_files, gcs_log_files):
        tf.io.gfile.copy(local_file, gcs_file)


copy_tensorboard_logs("./logs", OUTDIR + "/word2vec_logs")

埋め込みを可視化するには、Cloud Shell を開いて次のコマンドを実行します。

![embeddings_classifier_accuracy.png](assets/embeddings_classifier_accuracy.png)

## ラボ課題 4: 埋め込みの検索と分析

**ラボ課題 4**
`get_layer()` と `get_weights()` を使用してモデルから重みを取得します。`get_weights()` 関数は、レイヤーの重みを NumPy 配列として返します。

In [ ]:
# TODO 4a
weights = # ここにコードを記述してください
vocab = # ここにコードを記述してください

ベクトルとメタデータ ファイルを保存するために "word2vec" という名前のフォルダを作成します。

In [ ]:
!mkdir word2vec

ベクトルとメタデータ ファイルを作成して保存します。

In [ ]:
out_v = open("word2vec/vectors.tsv", "w", encoding="utf-8")
out_m = open("word2vec/metadata.tsv", "w", encoding="utf-8")

for index, word in enumerate(vocab):
    if index == 0:
        continue  # 0 はパディングのためスキップ
    vec = weights[index]
    out_v.write("\t".join([str(x) for x in vec]) + "\n")
    out_m.write(word + "\n")
out_v.close()
out_m.close()

`vectors.tsv` と `metadata.tsv` をローカル マシンにダウンロードし、[Embedding Projector](https://projector.tensorflow.org/) を開いて埋め込みを可視化します。

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.